# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/6b/2bnrv4k52z53kz5qmw40mf6r0000gp/T/ipykernel_48683/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [3]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [4]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [5]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [6]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:
To understand the importance of metadata a basic understanding of RAG is needed, I would describe the flow like this;
User Query is embedded -> Similarity search in vector database -> retrieve relevant chunks -> pass to LLM to generate answer grounded in retrieved context. 
Metadata can help this flow at two points:
1. The first is in retrieval where we can do some pre filtering. For example here if rather than having a single pdf we had multiple documents on cats we could ask for something like metadate.title == 'Cats in Australia health guidelines'. This would focus the retrieval on the subset of embeddings from the document 'Cats in Australia health guidelines' - essentially focussing and improving the relevance of retrieved chunks.
1. The second is once we have retrieved the relevant chunks, we can take the metadata and the chunks then add them to the context window so that the LLM and em can see the relevant sources of information as a reference. This allows us to verify and make sure that the chunks used in the RAG loop are relevant and actually exist, making for better traceability.
I see metadata like the spine of a book in a library - it will have information like a title, author, shelf number, genre - it doesn't directly affect the information I am after but it comes with the book and allows me to find it and understand where it came from.

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [7]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 135 chunks.
Chunk size: 1000 characters
Chunk overlap: 200 characters


In [8]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Chunk size = how much context each chunk has.
Chunk overlap = how much shared context there is between adjacent chunks. This can be thought of as continuity between chunks.

Chunk size:
1. Too little = a chunk might not have enough information to be useful.
2. Too much = a chunk might have too much information, where each chunk contains multiple ideas or concepts, essentially diluting the relevance of the chunk.

Chunk overlap:
1. Too little = important context might be lost at chunk boundaries, leading to less relevant retrieval results.
2. Too much = it can lead to redundant information across chunks, which may increase storage requirements and retrieval time without adding much value.

The best way to describe the tradeoff is precision vs context - smaller chunks with too little overlap may be more precise but miss important context, while larger chunks with too much overlap may provide richer context but be less precise. Plus the tuning of chunk size and overlap is highly dependent on the specific use case and the nature of the documents being processed. 

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [9]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [10]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [11]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.584 | page=8 | start_index=0
Detecting signs of pain or anxiety and evaluation of quality of life are most commonly of concern in the mature adult or senior cat but may be relevant at any life stage. During the physical examination, particular focus is on pain assessment and abdominal and thyroid palpation. A detailed mus- culoskeletal examination to detect signs of osteoarthr
--------------------------------------------------------------------------------
Source 2 | score=0.571 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 3 | score=0.565 | page=7 | sta

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

The similarity score measures how close two vectors are in embedding space. It is essentially telling you that there is a relationship between the query and the retrieved chunk based on their vector representations. A higher score indicates a stronger relationship, while a lower score suggests a weaker connection. The score is a relevance signal but not a truth signal. It provides ranked information for an LLM to reason over but does not guarantee an actual answer.

You can think of it like a search dog sniffing at an airport - the dog will sniff you if you have drugs in your luggage and sit if you do but it is just going off scent when in reality you just walked into a columbian party the night before but actually have nothing in your luggage. It is giving a signal to a match but is not a guarantee of the truth. 

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [12]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [13]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [14]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 8, score 0.584
Detecting signs of pain or anxiety and evaluation of quality of life
are most commonly of concern in the mature adult or senior cat but
may be relevant at any life stage.
During the physical examination, particular focus is on pain
assessment and abdominal and thyroid palpation. A detailed mus-
culoskeletal examination to detect signs of osteoarthritis is critical as
this condition is one of the most signi ﬁcant and underdiagnosed
diseases in cats.
23,28 A fundic examination is key to detecting signs of
ophthalmic disease or hypertension. 29 Practices should employ a
validated pain assessment scale or tool to diagnose, monitor, and
assist in the evaluation of patients for subtle signs of pain.
30
Changes in grooming habits, particularly increased grooming,
may signal a dermatologic issue such as atopy, food allergy, an
immune-mediated skin condition, infectious or parasitic disease,
endocrine condition, or paraneoplastic syndrom

In [15]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may suggest your cat needs veterinary attention include:

- **Pain or anxiety/stress behaviors** such as cowering, crouching, crawling, freezing, hiding, frantic fleeing, flattened or rotated ears, dilated pupils, tense body posture, or defensive vocalizing like hissing, growling, yowling, or screaming [Source 4]
- **Changes in grooming**, especially **increased grooming** or **reduced grooming** [Source 1][Source 2]
- **Reduced mobility** or signs of **joint pain/osteoarthritis** [Source 1][Source 2]
- **Changes in appetite, thirst, urination, vomiting, hairballs, or diarrhea** [Source 3]
- **Increased nocturnal activity or vocalization**, or other changes in normal habits or activity [Source 3]
- **House-soiling** or **aggression** toward people or other animals [Source 2]

If you’re seeing any of these signs, especially if they are new, worsening, or your cat seems uncomfortable, contact a veterinarian for an exam.

Sources:
{'source_label': 'Source 1', 'file': 'cat_healt

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [16]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("-" * 40)
    print("RETRIEVED CONTEXT (before generation):")
    retrieved = display_retrieval_results(question, k=answer_k)
    print()
    print("GENERATED ANSWER:")
    context = format_context(retrieved)
    print(rag_chain.invoke({"context": context, "question": question}))
    print()

QUESTION: What preventive care is recommended for cats?
----------------------------------------
RETRIEVED CONTEXT (before generation):
Source 1 | score=0.657 | page=15 | start_index=1549
17,120,121 This approach will eliminate existing infections, as well as decrease the risk of further infestation and subsequent associated clinical problems. Canine and feline housemates may be at risk of transmission of infectious parasites including roundworm and ﬂeas and therefore should be treated in synchronicity with newly acquired kittens or
--------------------------------------------------------------------------------
Source 2 | score=0.648 | page=2 | start_index=821
alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great an

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

I just slightly updated the vibe check code to return the retrieved context and the generated answer separately, this way we can see the k=4 top retrieved chunks for each question and get a better sense of the relevance of the retrieved context before it is passed to the LLM for synthesis and answer generation.
For the first 3 questions, the retrieved context seems relevant with similarity scores ranging from 0.4 to 0.6 and there abouts.These similarity scores are decent but not super high, which makes sense given the questions are somewhat general and the PDF is quite specific. The retrieved chunks for those questions contain information about cat health and life stages, which is relevant to the questions asked. Also since scores are not super high (especially for the question - What symptoms should make me call a veterinarian?) it shows how absolute score matters less than relative ranking within a query.
However the last question about taxes did not retrieve relevant context, even though the similarity score for the top retrieved chunks was around 0.380 to 0.365 compared to the other questions these scores are poorer. Suggesting that the retriever did not find a strong match in embedding space for the tax question, which makes sense since the PDF is about cat health/ life stages and not taxes. The key takeway for me is that retrieval doesn't know when a question is out of scope — it just returns the closest vectors regardless. The grounding in the system prompt is what prevented a hallucinated answer.

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

ANSWER
I will chose the question 'What symptoms should make me call a veterinarian?' since the retrieved context for this question had poorer similarity scores compared to the other relevant vibe check questions.


In [17]:
# Activity workspace — compare original vs tuned retrieval query
# Changing the wording of the retrieval query (easiest)

original_question = "What symptoms should make me call a veterinarian?"
updated_question  = "My cat is vomiting and has diarrhea, should I call a veterinarian?"

def run_retrieval_experiment(label: str, question: str, k: int) -> dict:
    print(f"\n{'#' * 100}")
    print(f"  {label}")
    print(f"  Query: {question}")
    print(f"{'#' * 100}")
    print("\n--- Retrieved Context ---")
    retrieved = display_retrieval_results(question, k=k)
    context = format_context(retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print("\n--- Generated Answer ---")
    print(answer)
    return {"retrieved": retrieved, "answer": answer}

original = run_retrieval_experiment("ORIGINAL", original_question, k=answer_k)
updated  = run_retrieval_experiment("UPDATED",  updated_question,  k=answer_k)

print(f"\n{'=' * 100}")
print("SCORE COMPARISON")
print(f"{'=' * 100}")
print(f"  {'Rank':<6} {'Original':>10}  {'Updated':>10}  {'Δ':>8}")
print(f"  {'-'*40}")
for i, ((_, score_o), (_, score_u)) in enumerate(
    zip(original["retrieved"], updated["retrieved"]), start=1
):
    diff = score_u - score_o
    diff_str = f"{'+' if diff >= 0 else ''}{diff:.3f}"
    print(f"  {i:<6} {score_o:.3f}{'':>8}  {score_u:.3f}{'':>6}  {diff_str}")


####################################################################################################
  ORIGINAL
  Query: What symptoms should make me call a veterinarian?
####################################################################################################

--- Retrieved Context ---
Source 1 | score=0.435 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 2 | score=0.404 | page=10 | start_index=4793
dysfunction syndrome, pain, hyper thyroidism, or hypertension. V et- erinary visits may be more challenging for the senior cat, in part be- cause many cat owners do 

In [18]:
# Activity workspace — compare original vs tuned retrieval query
# Changing the retrieval k (decreasing from 4 to 2)

activity_question = "What symptoms should make me call a veterinarian?"

original_k = 4
updated_k = 2
original = run_retrieval_experiment("ORIGINAL K=4", activity_question, k=original_k)
updated  = run_retrieval_experiment("UPDATED K=2",  activity_question, k=updated_k)


####################################################################################################
  ORIGINAL K=4
  Query: What symptoms should make me call a veterinarian?
####################################################################################################

--- Retrieved Context ---
Source 1 | score=0.435 | page=7 | start_index=2384
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and ask about any chronic enter- opathy or gastrointesti
--------------------------------------------------------------------------------
Source 2 | score=0.404 | page=10 | start_index=4793
dysfunction syndrome, pain, hyper thyroidism, or hypertension. V et- erinary visits may be more challenging for the senior cat, in part be- cause many cat owners

In [19]:
# Activity workspace — compare original vs tuned retrieval query
# Changing the chunk size

# Reuse the vector_store already built in Task 5 (chunk_size=1000, overlap=200)
# so original scores are identical to what was seen in Task 6.
updated_chunk_size = 200
chunk_overlap_ratio = 0.2  # keep overlap proportional at 20%

# Same question used throughout the activity
test_question = "What symptoms should make me call a veterinarian?"


def build_vector_store(chunk_size: int, overlap: int) -> QdrantVectorStore:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        add_start_index=True,
    )
    chunks = splitter.split_documents(pages)
    print(f"  chunk_size={chunk_size}, overlap={overlap} → {len(chunks)} chunks")
    return QdrantVectorStore.from_documents(
        documents=chunks,
        embedding=embeddings,
        location=":memory:",
        collection_name=f"cat_health_{chunk_size}",
        force_recreate=True,
    )


def display_results_from_store(store: QdrantVectorStore, query: str, k: int) -> list[tuple]:
    results = store.similarity_search_with_score(query, k=k)
    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        preview = doc.page_content[:300].replace("\n", " ")
        print(f"Source {index} | score={score:.3f} | page={page_display} | chars={len(doc.page_content)}")
        print(preview)
        print("-" * 80)
    return results


print("Building updated vector store...")
updated_store = build_vector_store(updated_chunk_size, int(updated_chunk_size * chunk_overlap_ratio))

print(f"\n{'#' * 100}")
print(f"  ORIGINAL  chunk_size={chunk_size}  (Task 5 vector_store, chunk_overlap={chunk_overlap})")
print(f"  Query: {test_question}")
print(f"{'#' * 100}")
original_results = display_results_from_store(vector_store, test_question, k=answer_k)
original_answer = rag_chain.invoke({"context": format_context(original_results), "question": test_question})
print("\n--- Generated Answer ---")
print(original_answer)

print(f"\n{'#' * 100}")
print(f"  UPDATED  chunk_size={updated_chunk_size}")
print(f"  Query: {test_question}")
print(f"{'#' * 100}")
updated_results = display_results_from_store(updated_store, test_question, k=answer_k)
updated_answer = rag_chain.invoke({"context": format_context(updated_results), "question": test_question})
print("\n--- Generated Answer ---")
print(updated_answer)

print(f"\n{'=' * 100}")
print("SCORE COMPARISON")
print(f"{'=' * 100}")
print(f"  {'Rank':<6} {f'Original ({chunk_size})':>16}  {f'Updated ({updated_chunk_size})':>15}  {'Δ':>8}")
print(f"  {'-'*50}")
for i, ((_, score_o), (_, score_u)) in enumerate(
    zip(original_results, updated_results), start=1
):
    diff = score_u - score_o
    diff_str = f"{'+' if diff >= 0 else ''}{diff:.3f}"
    print(f"  {i:<6} {score_o:.3f}{'':>12}  {score_u:.3f}{'':>10}  {diff_str}")

Building updated vector store...
  chunk_size=200, overlap=40 → 662 chunks

####################################################################################################
  ORIGINAL  chunk_size=1000  (Task 5 vector_store, chunk_overlap=200)
  Query: What symptoms should make me call a veterinarian?
####################################################################################################
Source 1 | score=0.435 | page=7 | chars=987
Asking speci ﬁc questions concerning whether vomiting, vom- iting hairballs, or diarrhea is occurring, and the frequency of each, is recommended as some clients may consider vomiting or vomiting hairballs to be normal for their cat. Additionally, discuss the im- portance of monitoring weight, and as
--------------------------------------------------------------------------------
Source 2 | score=0.404 | page=10 | chars=992
dysfunction syndrome, pain, hyper thyroidism, or hypertension. V et- erinary visits may be more challenging for the senior

### 🏗️ Activity Notes

Three retrieval settings were tuned against the question *"What symptoms should make me call a veterinarian?"* — chosen because it returned the weakest scores in the vibe check (top score 0.435, vs ~0.58 for more explicit queries).

---

#### Experiment 1 — Query wording

| Rank | Original | Specific symptom query | Δ |
|------|----------|----------------------|---|
| 1 | 0.435 | 0.450 | +0.016 |
| 2 | 0.404 | 0.404 | +0.000 |
| 3 | 0.400 | 0.373 | −0.027 |
| 4 | 0.390 | 0.371 | −0.020 |

**Original:** "What symptoms should make me call a veterinarian?"  
**Updated:** "My cat is vomiting and has diarrhea, should I call a veterinarian?"

The concrete symptom query pulled rank 1 up by +0.016 by landing the embedding closer to the chunk that explicitly mentions vomiting and diarrhea. However ranks 3 and 4 dropped — the specificity narrowed the semantic reach and missed broader symptom chunks the general phrasing caught. The generated answer improved in directness: it moved from a general list to "Yes — vomiting and diarrhea are specifically listed as signs to discuss with a vet."

**Takeaway:** Specific queries improve the top hit but reduce breadth. Best when the user already knows the symptom; less useful for open-ended "what should I watch for" questions.

---

#### Experiment 2 — Retrieval k (k=4 → k=2)

| Rank | k=4 | k=2 |
|------|-----|-----|
| 1 | 0.435 | 0.435 |
| 2 | 0.404 | 0.404 |
| 3 | 0.400 | — |
| 4 | 0.390 | — |

**k=4 answer:** Listed changes in appetite, urination/thirst, vomiting, diarrhea, nocturnal activity, vocalization, and changes in habits — citing Sources 1–3 for disease/pain/cognitive dysfunction context, with Source 4 (kitten behaviour, score 0.390) adding minor noise.

**k=2 answer:** Covered the same core symptoms from Sources 1–2 and still mentioned senior cat specifics (reduced jumping/climbing). The answer was comparably complete despite having half the context, because the top 2 chunks already contained the most relevant content.

The key observation: Source 4 at rank 4 (score 0.390, about kitten behaviour counselling) wasn't actually useful for this question — removing it by dropping to k=2 didn't hurt the answer at all.

**Takeaway:** Reducing k can sharpen answers when the top chunks are strong and lower-ranked chunks add noise. The score gap between rank 2 (0.404) and rank 3 (0.400) is tiny here — both chunks are nearly equally relevant, so k=4 didn't hurt. But at k=4 rank 4 was clearly off-topic.

---

#### Experiment 3 — Chunk size (1000 → 200 characters)

| Rank | chunk_size=1000 | chunk_size=200 | Δ |
|------|----------------|---------------|---|
| 1 | 0.435 | 0.518 | +0.083 |
| 2 | 0.404 | 0.430 | +0.026 |
| 3 | 0.400 | 0.425 | +0.025 |
| 4 | 0.390 | 0.417 | +0.027 |

135 chunks (1000-char) → 662 chunks (200-char). The scores improved significantly — top hit up by +0.083 — but the **answer got worse**. The 200-char chunks retrieved were narrow: urinary tract urgency, DJD in senior cats, litter box elimination, soiling behaviour. These are topically adjacent ("call a vet") but missed the broad symptom list the question was actually asking for. The 1000-char chunk that contained the full paragraph on vomiting, diarrhea, nocturnal activity, and weight monitoring didn't appear at all in the 200-char top-4.

**k=1000 answer:** "Changes in appetite, increased urination, vomiting, diarrhea, nocturnal activity, vocalization, changes in habits or activity" — a direct, comprehensive list.  
**k=200 answer:** "Seek veterinary help promptly for elimination problems such as urinary tract issues or soiling behaviour... owners should learn to read body language" — specific but narrow, misses the main symptom list.

**Takeaway:** Higher retrieval scores do not guarantee a better answer. Smaller chunks scored higher because each vector was tightly focused on one idea ("call the vet for urinary issues"), making it a strong embedding match. But the 1000-char chunk, despite a lower score, held the complete symptom list the LLM needed. This is the precision vs. context tradeoff in action — and it shows why evaluating chunk size on retrieval scores alone is misleading. Answer quality must be assessed too.

---

#### Overall synthesis

Across all three experiments the same tension appears: **precision vs. coverage**. A specific query, low k, and small chunks all push toward precision — better top-hit scores, more focused answers. But Experiment 3 shows precision can backfire: the 200-char chunks scored highest yet produced the least useful answer, because the relevant content was spread across a paragraph that got split apart. The most important lesson is that **retrieval score is a proxy for answer quality, not a guarantee of it**. For this cat health PDF, chunk_size ~1000 with k=4 and a moderately general query gave the best end-to-end results across all three experiments.

This synthesis was written with the help of Claude - but I double checked to make sure it was accurate and made sense.